# 03 — Batch & Composite Experiments

`QickworkspaceV2` provides two composite types:

| Class | Behaviour |
|---|---|
| `BatchExperiment` | Runs experiments **sequentially**, in order.  Each can feed results into the next. |
| `ParallelExperiment` | Runs experiments **in parallel** (separate threads).  Use when experiments are independent. |

Both return a `dict[name → ExperimentData]`.

In [ ]:
import sys; sys.path.insert(0, '../')

from QickworkspaceV2 import SimulatedBackend, ExperimentConfig, BatchExperiment, ParallelExperiment
from QickworkspaceV2.experiments.resonator import ResonatorSpec
from QickworkspaceV2.experiments.qubit_ge  import QubitSpec, PowerRabi
from QickworkspaceV2.experiments.coherence import T1, Ramsey

backend = SimulatedBackend(noise_level=0.015)
backend.activate()

config_list = [{
    "name": "Q1",
    "ch":  {"ro_ch": 0, "res_ch": 0, "qb_ch": 1},
    "res": {"res_freq_ge": 6700.0, "res_gain": 0.5, "res_length": 2.0},
    "qb":  {"qb_freq_ge": 5000.0, "pi_gain_ge": 0.5, "sigma": 0.025},
    "reps": 100, "relax_delay": 300.0, "steps": 51,
    "kappa": 4.0, "qb_kappa": 3.0,
    "wait_time_start": 0.0, "wait_time_stop": 150.0,
    "T1_true": 50.0, "T2_true": 20.0, "ramsey_freq": 1.5,
}]
cfg_all = ExperimentConfig(config_list)
cfg = cfg_all.get_qubit('Q1')

## BatchExperiment — sequential pipeline

Each entry is `(name, experiment_instance)`.  Results are available immediately
after the batch completes as `results[name]`.

In [ ]:
batch = BatchExperiment([
    ('res_spec',    ResonatorSpec(cfg, backend=backend)),
    ('qubit_spec',  QubitSpec(cfg, backend=backend)),
    ('power_rabi',  PowerRabi(cfg, backend=backend)),
])

results = batch.run(py_avg=5)
print('Completed steps:', list(results.keys()))

In [ ]:
# Access individual results by name
for name, r in results.items():
    val = f'{r.scalar_result:.4f}' if r.scalar_result is not None else 'N/A'
    print(f'{name:<15} quality={r.quality.value:<12} scalar={val}')

## Feeding results between steps

The cleanest pattern: update `cfg_all` after each step so later steps use
the freshly calibrated values.

In [ ]:
from QickworkspaceV2 import CalibrationStore
import tempfile, os

store = CalibrationStore(os.path.join(tempfile.gettempdir(), 'batch_demo.json'))

def update(key, result):
    """Helper: write scalar_result back to live config + store."""
    val = result.scalar_result
    if val is not None:
        cfg_all.update(key, val, q_index='Q1')
        store.set('Q1', key, val)
        print(f'  updated {key} = {val:.4f}')

# Manual sequential pipeline with intermediate updates
print('--- res_spec ---')
r = ResonatorSpec(cfg_all.get_qubit('Q1'), backend=backend).run(py_avg=5)
update('res_freq_ge', r)

print('--- qubit_spec ---')
r = QubitSpec(cfg_all.get_qubit('Q1'), backend=backend).run(py_avg=5)
update('qb_freq_ge', r)

print('--- power_rabi ---')
r = PowerRabi(cfg_all.get_qubit('Q1'), backend=backend).run(py_avg=5)
update('pi_gain_ge', r)

print('\nStore after pipeline:')
print(store.summary('Q1'))

## ParallelExperiment — independent experiments at the same time

Useful when two experiments don't share state and you want to save wall time.
e.g., T1 and Ramsey use the same (already calibrated) π-pulse, so they can run together.

In [ ]:
cfg_cal = cfg_all.get_qubit('Q1')   # already has updated freq + gain

parallel = ParallelExperiment([
    ('t1',     T1(cfg_cal,     backend=backend)),
    ('ramsey', Ramsey(cfg_cal, backend=backend)),
])

par_results = parallel.run(py_avg=5)

T1_us  = par_results['t1'].scalar_result
T2r_us = par_results['ramsey'].scalar_result
print(f'T1  = {T1_us:.1f} µs' if T1_us  else 'T1 fit failed')
print(f'T2* = {T2r_us:.1f} µs' if T2r_us else 'Ramsey fit failed')

**Next:** [04_auto_calibrate.ipynb](04_auto_calibrate.ipynb) — fully automated calibration with `AutoCalibrate`.